# Init

In [ ]:
# init
from importlib import reload
from pprint import pprint
import os
import sys
from pathlib import Path
import pandas as pd

def reloadPckg():
    reload(tgm)
    reload(too)
    reload(osmt)
    reload(tph)

sys.path.append('..')
sys.path.append('../tools')
devPath = "C:/Users/gonta/D/software developer/"
sys.path.append(os.path.join(devPath, "packages"))


import tests.osm_test as osmt
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph

# Cleaning

## * import

In [1]:
data_dir = Path(os.path.join(os.getcwd(), '..', 'data', 'raw', 'osm countries queries'))
files_dirs = [f for f in data_dir.glob('*')]
print(len(files_dirs))

raw_by_cntr = {}

# for chunks and non chunk files
for dir in files_dirs:
    files_elements = [tgm.load(str(f))['elements'] for f in dir.glob('*.json')]
    elements = [ele for list in files_elements for ele in list]
    raw_by_cntr[str(dir.name)] = elements

print(len(raw_by_cntr))

83
83


In [4]:
path = Path(os.path.join(os.getcwd(),'../data/normalized'))
processsed_countries = [f.name for f in path.glob('*')]
print(len(processsed_countries))
# to_process_raw_by_cntr = {k:v for k,v in raw_by_cntr.items() if k not in processsed_countries}

70


In [13]:
to_process_raw_by_cntr = raw_by_cntr
print(len(to_process_raw_by_cntr))

83


In [8]:
resume = {}
for country, raw in raw_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl_1'] = len([ele for ele in raw if ele['tags']['admin_level'] == '4'])
    resume[country]['lvl_2'] = len([ele for ele in raw if ele['tags']['admin_level'] == '6'])
    resume[country]['lvl_3'] = len([ele for ele in raw if ele['tags']['admin_level'] == '8'])
    resume[country]['total'] = len([ele for ele in raw if ele['tags']['admin_level'] != '2'])

resume = pd.DataFrame(resume)
table = resume.T
tph.df_peek(table)

,lvl_1,lvl_2,lvl_3,total
Afghanistan,34,0,0,34
AkrotiriAndDhekelia,0,0,0,0
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Armenia,11,15,0,26


## * use sovereign countries only

In [9]:
sovereign_countries = tgm.load(os.path.join('..', 'data', 'sovereign countries.json'))
len(sovereign_countries)

214

In [10]:
print(len(raw_by_cntr))
raw_by_cntr = {k:df for k,df in raw_by_cntr.items() if k in sovereign_countries}
len(raw_by_cntr)

83


70

## * clean countries

In [11]:
import copy
def clean_country_data(raw_by_cntr):

    cleaned_by_cntr = copy.deepcopy(raw_by_cntr)  

    for country, raw_data in cleaned_by_cntr.items():

        # convert id to string
        for ele in raw_data:
            ele['id'] = str(ele['id'])

        # add parent_id to level 4 entities
        cntr_id = list(filter(lambda ele: ele['tags']['admin_level'] == '2', raw_data))[0]['id']
        for ele in raw_data:
            if ele['tags']['admin_level'] == '4':
                ele['tags']['parent_id'] = str(cntr_id)

        # add country_name and country_id tag
        for ele in raw_data:
            ele['tags']['country_name'] = country
            ele['tags']['country_id'] = cntr_id

    return cleaned_by_cntr

In [12]:
cleaned_by_cntr = clean_country_data(raw_by_cntr)
print(len(cleaned_by_cntr))

70


In [13]:
temp = [ele['tags'].get('parent_id',None) for k,v in cleaned_by_cntr.items() for ele in v]
pd.Series(temp).map(type).value_counts()

<class 'str'>         120018
<class 'NoneType'>        70
Name: count, dtype: int64

## * convert to dataframe

In [14]:
df_by_cntr = {k:too.normalizeOSM(elems) for k,elems in cleaned_by_cntr.items()}
len(df_by_cntr)

70

In [15]:
combined = pd.concat(df_by_cntr.values(), ignore_index=True)
combined.map(type).stack().value_counts()

<class 'pandas._libs.missing.NAType'>    1058618
<class 'str'>                             862790
Name: count, dtype: int64

## * save

In [16]:
print(len(df_by_cntr))

70


In [19]:
# save data by country
for k,df in df_by_cntr.items():
    path = os.path.join('../data/normalized', k, f'{k}_normalized.pkl')
    tgm.dump(path, df)

## * review

In [20]:
# import
dataDir = Path(os.path.join(os.getcwd(), '..', 'data/normalized'))
rawFiles = [f for f in dataDir.rglob('*.pkl')]
print(len(rawFiles))

df_by_cntr = {
    file.parent.name:tgm.load(str(file))
    for file in rawFiles
}
print(len(df_by_cntr))

70
70


In [21]:
for k,df in df_by_cntr.items():
    cntr = df[df['tags.admin_level']=='2']
    print([k, cntr['tags.name'].iloc[0], cntr['tags.name:en'].iloc[0]])

['Afghanistan', 'افغانستان', 'Afghanistan']
['Albania', 'Shqipëria', 'Albania']
['Algeria', 'Algérie ⵍⵣⵣⴰⵢⴻⵔ الجزائر', 'Algeria']
['Andorra', 'Andorra', 'Andorra']
['Angola', 'Angola', 'Angola']
['Anguilla', 'Anguilla', 'Anguilla']
['AntiguaAndBarbuda', 'Antigua and Barbuda', 'Antigua and Barbuda']
['Argentina', 'Argentina', 'Argentina']
['Armenia', 'Հայաստան', 'Armenia']
['Australia', 'Australia', 'Australia']
['Austria', 'Österreich', 'Austria']
['Bangladesh', 'বাংলাদেশ', 'Bangladesh']
['Belarus', 'Беларусь', 'Belarus']
['Belgium', 'België / Belgique / Belgien', 'Belgium']
['Bhutan', 'འབྲུག་ཡུལ།', 'Bhutan']
['Brazil', 'Brasil', 'Brazil']
['Bulgaria', 'България', 'Bulgaria']
['Cambodia', 'ព្រះរាជាណាចក្រ\u200bកម្ពុជា', 'Cambodia']
['Chile', 'Chile', 'Chile']
['Colombia', 'Colombia', 'Colombia']
['Czechia', 'Česko', 'Czechia']
['Denmark', 'Danmark', 'Denmark']
['Ecuador', 'Ecuador', 'Ecuador']
['Estonia', 'Eesti', 'Estonia']
['Eswatini', 'Eswatini', 'Eswatini']
['FaroeIslands', 'Føroyar

In [22]:
resume = {}
for country, df in df_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl1'] = len(df[df['tags.admin_level'] == '4'])
    resume[country]['lvl2'] = len(df[df['tags.admin_level'] == '6'])
    resume[country]['lvl3'] = len(df[df['tags.admin_level'] == '8'])
    resume[country]['total'] = len(df[df['tags.admin_level'] != '2'])

resume = pd.DataFrame(resume)
table = resume.T
tph.df_peek(table)

,lvl1,lvl2,lvl3,total
Afghanistan,34,0,0,34
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Armenia,11,15,0,26
Australia,15,600,0,615


## 3. osm duplicates test

### * check data

In [122]:
dups_id = df_all[df_all.duplicated(subset=['id'])]['id'].to_list()
print(len(dups_id))

dups_df = df_all[df_all.duplicated(subset=['id'], keep=False)]
print(len(dups_df))

dups_id_tuples = dups_df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print(len(dups_id_tuples))

1104
1636
1636


In [41]:
dups_id_parent = df_all[df_all.duplicated(subset=['id','tags.parent_id'], keep=False)][['id','tags.parent_id']].apply(tuple, axis=1).to_list()
print(len(dups_id_parent))
dups_id_parent_cntr = df_all[df_all.duplicated(subset=['id','tags.parent_id','tags.country_id'], keep=False)][['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print(len(dups_id_parent_cntr))

1380
0


In [55]:
df_all[(df_all[['id','tags.parent_id','tags.country_id']] == ('72639', '60199', '60199')).all(axis=1)]

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:us,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
113807,relation,72639,4,60199,Автономна Республіка Крим,<NA>,<NA>,UA-43,<NA>,<NA>,<NA>,<NA>,<NA>,Ukraine,60199


In [ ]:
all_id_tuples = df_all[['id','tags.parent_id','tags.country_id']].fillna('#').apply(tuple, axis=1).to_list()
print(len(all_id_tuples))

In [102]:
to_review_ids = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065')]

In [103]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"),to_review_ids)

In [51]:
temp1 = too.is_child_inside_parent('6543282','6543265')
[[k,v['result']] for k,v in temp1.items()]

[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 7934379969)
[INFO] :     * Finished testing (admin_centre): False
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 7934379969)
[INFO] :     * Finished testing (place): False
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 4298631214)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 36.8944071, lon: 6.4872707)
[INFO] :     * Finished testing (centroid): True


[['admin_centre', False],
 ['label', None],
 ['place', False],
 ['geom_node', True],
 ['centroid', True]]

In [37]:
tph.get_from_df(df_all,['id','tags.parent_id','tags.country_id'],('6543282', '6543265', '192756'))

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:us,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
2462,rel,6543282,8,6543265,Beni Zid ⴱⴻⵏⵉ ⵣⵉⴷ بنى زيد,<NA>,<NA>,<NA>,Algeria,<NA>,<NA>,<NA>,<NA>,Algeria,192756


In [32]:
dups_id_tuples

[('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'),
 ('14326718', '3824588', '184640'),
 ('14326687', '3825003', '184640'),
 ('14326687', '3921322', '184640'),
 ('14326718', '3921322', '184640'),
 ('59190', '59752', '59065'),
 ('59190', '59195', '59065'),
 ('5598639', '70809', '59065'),
 ('8030910', '70809', '59065'),
 ('5598639', '70802', '59065'),
 ('8030910', '70802', '59065'),
 ('8075978', '70671', '59065'),
 ('8144265', '69554', '59065'),
 ('8144265', '79911', '59065'),
 ('8075978', '6825778', '59065'),
 ('7400', '4217435', '52411'),
 ('416271', '4217435', '52411'),
 ('7400', '53134', '52411'),
 ('416271', '53134', '52411'),
 ('117571', '7400', '52411'),
 ('1561101', '7400', '52411'),
 ('117571', '416271', '52411'),
 ('1561101', '416271', '52411'),
 ('13482954', '12171908', '184629'),
 ('13482954', '12171907', '184629'),
 ('406273', '405935', '59470'),
 ('406273', '325866', '59470'),
 ('2662003', '362413', '59470'),
 ('2662004', '302819', '59470'),
 ('2662005', '

### * select ents and run test

In [45]:
dups_test_processed = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_test_processed.pkl"))
print(len(dups_test_processed))

1634


In [44]:
print('dups_df: ', len(dups_df))
print('dups_test_processed: ', len(dups_test_processed))
to_test = dups_df[['id', 'tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(dups_test_processed)
to_test = dups_df[~to_test]
print(len(to_test))
# to_test = list(to_test.groupby('tags.country_id'))
print(len(to_test))
# to_test = to_test[:2]
# print(len(to_test))

dups_df:  1636
dups_test_processed:  0
1636
1636


In [40]:
def apply_to_group(df):
    country_name = df.iloc[0]['tags.country_name']
    path = os.path.join(os.getcwd(), r"..\data\tests results\osm duplicates test", country_name, f"{country_name}_dups_test_res.pkl")
    osmt.osm_test_center(
    df,
    save_temp=True,
    save_path=path
    )

In [ ]:
# [OLD]
# to_test.groupby('tags.country_id').apply(apply_to_group)

[INFO] :  ^ [2/2]: testing ('6543282', '6543273', '192756'):
[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 7934379969)
[INFO] :     * Finished testing (admin_centre): True
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 7934379969)
[INFO] :     * Finished testing (place): True
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 4298631214)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 36.8944071, lon: 6.4872707)
[INFO] :     * Finished testing (centroid): False
[INFO] :   * saving ...
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'missing', 'place': 'ok', 'geom_node': 'ok', 'centroid': 'ok'}
C:\Users\gonta\AppDa

""


In [53]:
for key, group in to_test:
    apply_to_group(group)

[INFO] :  ^ [2/2]: testing ('18305258', '19377767', '49915'):
[INFO] :    > Getting admin_centre:
[INFO] :     * Missing node (admin_centre)
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Missing node (place)
[INFO] :    > Getting geom_node:
[INFO] :     * Missing node (geom_node)
[INFO] :    > Getting centroid:
[INFO] :     * Missing node (centroid)
[INFO] :   * saving ...
[INFO] :  $ finished: status: {'admin_centre': 'missing', 'label': 'missing', 'place': 'missing', 'geom_node': 'missing', 'centroid': 'missing'}


### * update processed data

In [48]:
path_obj = Path(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test"))
test_res_list = [tgm.load(file) for file in list(path_obj.rglob('*/*.pkl'))]
test_res = {}
for d in test_res_list:
    test_res.update(d)
print(len(test_res))

del test_res_list

1636


In [ ]:
# test_res = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_test_res.pkl"))
# print(len(test_res))

2


In [57]:
dups_test_processed = set()
for tuple_id, res in test_res.items():
    if 'ok' in [ele['status'] for ele in res.values()]:
        dups_test_processed.add(tuple_id)
print(len(dups_test_processed))

1634


In [44]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_test_processed.pkl"), dups_test_processed)

### * review

In [18]:
path_obj = Path(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test"))
test_res_list = [tgm.load(file) for file in list(path_obj.rglob('*/*.pkl'))]
test_res = {}
for d in test_res_list:
    test_res.update(d)
print(len(test_res))

del test_res_list

1636


In [19]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1399
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                          55
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               20
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [20]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6242
missing    1938
Name: count, dtype: int64

### * select false entities and childs to delete

In [21]:
test_res_simplified = {}
for id, val in test_res.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # select the first one that gives True
            # test_res_simplified[id] = False
            # if val[type]['result'] is True:
            #     test_res_simplified[id] = val[type]['result']
            #     break

            # assign the first valid result
            # test_res_simplified[id] = val[type]['result']
            # break

            # make a voting weight selection
            if val[type]['result'] is True: true_count += 1
            else: false_count += 1
    
    test_res_simplified[id] = true_count >= false_count


pd.Series([v for k,v in test_res_simplified.items()]).value_counts()

True     1518
False     118
Name: count, dtype: int64

In [69]:
dups_tuples_to_delete_parents = [k for k,v in test_res_simplified.items() if v is False]
print(len(dups_tuples_to_delete_parents))

118


In [73]:
dups_tuples_to_delete_parents

[('14326718', '3921322', '184640'),
 ('59190', '59752', '59065'),
 ('5598639', '70802', '59065'),
 ('8030910', '70802', '59065'),
 ('416271', '4217435', '52411'),
 ('7400', '53134', '52411'),
 ('1561101', '7400', '52411'),
 ('117571', '416271', '52411'),
 ('13482954', '12171907', '184629'),
 ('406273', '325866', '59470'),
 ('2662005', '334443', '59470'),
 ('2662004', '332924', '59470'),
 ('4283101', '3759431', '186382'),
 ('4283101', '3759439', '186382'),
 ('17755316', '3759439', '186382'),
 ('4283101', '3759447', '186382'),
 ('4283101', '3759438', '186382'),
 ('4283101', '3759448', '186382'),
 ('4283101', '3759427', '186382'),
 ('17755317', '3759427', '186382'),
 ('17755318', '3759427', '186382'),
 ('17759455', '3759427', '186382'),
 ('1980749', '153549', '167454'),
 ('7128807', '1310024', '120027'),
 ('11908912', '1342117', '120027'),
 ('11919498', '1342124', '120027'),
 ('18963551', '1343276', '120027'),
 ('11909897', '1438574', '120027'),
 ('11909898', '1438574', '120027'),
 ('1346

In [104]:
temp = [[ele[0],ele[2]] for ele in dups_tuples_to_delete_parents]
dups_tuples_to_delete_childs = [ele for ele in all_id_tuples if [ele[1],ele[2]] in temp]
print(len(dups_tuples_to_delete_childs))

# dups_tuples_to_delete = dups_tuples_to_delete_childs + dups_tuples_to_delete_parents
dups_tuples_to_delete = dups_tuples_to_delete_parents
print(len(dups_tuples_to_delete))

923
118


In [105]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

### * add manual review deletion

In [106]:
to_review_ids = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "to_review_ids.pkl"))
print(len(to_review_ids))

6


In [116]:
to_review_ids

[('72639', '60189', '60189'),
 ('72639', '60199', '60199'),
 ('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'),
 ('59190', '59195', '59065'),
 ('59190', '59752', '59065')]

In [115]:
[[k,v['result']] for k,v in test_res[('6543282', '6543265', '192756')].items()]

[['admin_centre', False],
 ['label', None],
 ['place', False],
 ['geom_node', True],
 ['centroid', True]]

In [110]:
manual_dups_tuples_to_delete = [('6543282', '6543273', '192756'), ('59190', '59195', '59065')]

In [111]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "manual_dups_tuples_to_delete.pkl"), manual_dups_tuples_to_delete)

In [112]:
dups_tuples_to_delete = dups_tuples_to_delete + manual_dups_tuples_to_delete
print(len(dups_tuples_to_delete))

120


In [113]:
tgm.dump(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"), dups_tuples_to_delete)

In [129]:
print(len(dups_id_tuples))
print(len(dups_tuples_to_delete))
temp = tgm.complement(dups_id_tuples, dups_tuples_to_delete)
print(len(temp))

1636
120
1516
